In [0]:

# Phase 5 - Step 2: Validate Customer IDs

from pyspark.sql import functions as F

# Read the Silver Delta table
silver_table = "workspace.customer_data_pipeline.silver_customers"

df_silver = spark.table(silver_table)

# Count total records
total_records = df_silver.count()

# Count missing customer IDs
null_customer_ids = df_silver.filter(
    F.col("customer_id").isNull()
).count()

print("Total Silver records:", total_records)
print("Missing customer IDs:", null_customer_ids)

# Fail validation if customer IDs are missing
assert null_customer_ids == 0, "DATA QUALITY FAILED: Missing customer IDs"

print("CUSTOMER ID VALIDATION PASSED")


In [0]:

# Phase 5 - Step 3: Detect Duplicate Customer IDs

from pyspark.sql import functions as F

# Group by customer ID and find duplicates
df_duplicates = (
    df_silver
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

# Count duplicate customer ID groups
duplicate_count = df_duplicates.count()

print("Duplicate customer IDs:", duplicate_count)

# Display duplicates, if any
display(df_duplicates)

# Fail validation if duplicates exist
assert duplicate_count == 0, "DATA QUALITY FAILED: Duplicate customer IDs found"

print("DUPLICATE CUSTOMER ID VALIDATION PASSED")


In [0]:

# Phase 5 - Step 4: Reusable Null Validation Function

from pyspark.sql import functions as F

def validate_not_null(df, column_name):
    """
    Check whether a column contains null values.
    Raise an error if any null values exist.
    """

    null_count = df.filter(
        F.col(column_name).isNull()
    ).count()

    print(f"Column: {column_name}")
    print(f"Null records: {null_count}")

    assert null_count == 0, (
        f"DATA QUALITY FAILED: {column_name} contains null values"
    )

    print(f"VALIDATION PASSED: {column_name}")


# Test our reusable function
validate_not_null(df_silver, "customer_id")
validate_not_null(df_silver, "customer_name")


In [0]:

# Phase 5 - Step 5: Reusable Duplicate Validation

from pyspark.sql import functions as F

def validate_unique(df, column_name):
    """
    Validate that a column contains unique values.
    Raise an error if duplicate values are found.
    """

    duplicate_groups = (
        df.groupBy(column_name)
        .count()
        .filter(F.col("count") > 1)
    )

    duplicate_count = duplicate_groups.count()

    print(f"Column: {column_name}")
    print(f"Duplicate groups: {duplicate_count}")

    assert duplicate_count == 0, (
        f"DATA QUALITY FAILED: Duplicate values in {column_name}"
    )

    print(f"VALIDATION PASSED: {column_name}")


# Test our reusable function
validate_unique(df_silver, "customer_id")


In [0]:

# Phase 5 - Step 6: Bronze-Silver-Gold Reconciliation

from pyspark.sql import functions as F

# Read Delta tables
df_bronze = spark.table(
    "workspace.customer_data_pipeline.bronze_customers"
)

df_silver = spark.table(
    "workspace.customer_data_pipeline.silver_customers"
)

df_gold = spark.table(
    "workspace.customer_data_pipeline.gold_customers_by_city"
)

# Count records
bronze_count = df_bronze.count()
silver_count = df_silver.count()

# Count unique customers in Bronze
bronze_unique = (
    df_bronze
    .select("customer_id")
    .distinct()
    .count()
)

# Calculate Gold customer total
gold_total = df_gold.agg(
    F.sum("total_customers")
).first()[0]

print("Bronze raw records:", bronze_count)
print("Bronze unique customers:", bronze_unique)
print("Silver clean records:", silver_count)
print("Gold total customers:", gold_total)

# Reconciliation checks
assert bronze_unique == silver_count, (
    "FAILED: Bronze and Silver mismatch"
)

assert silver_count == gold_total, (
    "FAILED: Silver and Gold mismatch"
)

print("BRONZE-SILVER-GOLD RECONCILIATION PASSED")
